# Module 18 — Exercises

These exercises follow the order of the concept guide: exact samplers, rejection and importance sampling,
Markov chains, Metropolis–Hastings, the VAE's Gaussian KL, the diffusion forward process and the score, and
finally a Bayesian mini-project with MCMC. Several checks compare against *hidden* values of seeded
computations, so follow the specified recipe (which `rng` call, in which order) exactly.

**How these exercises work.** Each coding exercise has a *starter* cell (replace `# YOUR CODE HERE` with your implementation) followed by a *check* cell. Run the check cell to get an instant ✅/❌. Checks never stop the notebook, so you can run everything top-to-bottom at any time and see your score at the end. Some checks compare against a *hidden* answer (only a hash is stored), so you cannot peek at the number — think first, then verify. Conceptual questions hide their answer behind a *Show answer* toggle. Mini-projects are open-ended and come with a rubric. Difficulty is marked ★☆☆ (warm-up), ★★☆ (core), ★★★ (stretch).

### Setup and self-check helpers
Run the next two cells first.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
rng = np.random.default_rng(0)
plt.rcParams["figure.figsize"] = (7, 4)

In [ ]:
import hashlib as _hashlib
import numpy as _np

_results = {}

def _canon(value, decimals=3):
    """Canonical string form of an answer, so hashes are stable."""
    if isinstance(value, (bool, _np.bool_)):
        return "true" if value else "false"
    if isinstance(value, str):
        return " ".join(value.strip().lower().split())
    if isinstance(value, (int, _np.integer)):
        return str(int(value))
    if isinstance(value, (float, _np.floating)):
        s = f"{float(value):.{decimals}f}"
        return "0." + "0" * decimals if s in ("-0." + "0" * decimals, "-0") else s
    if isinstance(value, dict):
        return "{" + ",".join(f"{_canon(k, decimals)}:{_canon(v, decimals)}" for k, v in sorted(value.items(), key=lambda kv: str(kv[0]))) + "}"
    try:
        arr = _np.asarray(value)
        if arr.ndim == 0:
            return _canon(arr.item(), decimals)
        return "[" + ",".join(_canon(x, decimals) for x in arr.ravel().tolist()) + "]"
    except Exception:
        return str(value)

def sha_of(value, decimals=3):
    return _hashlib.sha256(_canon(value, decimals).encode()).hexdigest()[:16]

def check(name, condition, hint=""):
    """Record and print a pass/fail mark. Never raises, so the notebook keeps running."""
    try:
        ok = bool(condition)
    except Exception:
        ok = False
    _results[name] = ok
    print(("✅ " if ok else "❌ ") + name + ("" if ok or not hint else f"   — hint: {hint}"))
    return ok

def check_close(name, got, expected, tol=1e-6, hint=""):
    try:
        g = _np.asarray(got, dtype=float)
        e = _np.asarray(expected, dtype=float)
        ok = g.shape == e.shape and bool(_np.allclose(g, e, atol=tol, rtol=tol))
    except Exception:
        ok = False
    return check(name, ok, hint)

def check_hash(name, value, expected_sha, decimals=3, hint=""):
    """Compare an answer against a hidden expected value (only its hash is stored)."""
    try:
        ok = value is not None and sha_of(value, decimals) == expected_sha
    except Exception:
        ok = False
    return check(name, ok, hint)

def summary():
    n_ok = sum(1 for v in _results.values() if v)
    n = len(_results)
    print(f"\nScore: {n_ok}/{n} checks passed" + ("  🎉" if n and n_ok == n else ""))
    if n and n_ok < n:
        print("Not yet passing: " + ", ".join(k for k, v in _results.items() if not v))

## Exercise 1 — Exact samplers: inverse CDF and cumulative sums  `★☆☆`

Implement two samplers that use **only** `rng.random` (uniforms) plus NumPy arithmetic — no
`rng.exponential`, no `rng.choice`:

* `sample_exponential(lam, n, rng)` → array of `n` draws from Exponential($\lambda$) via the inverse CDF
  $X = -\ln(1 - U)/\lambda$, where `U = rng.random(n)` is drawn **once, in that form**;
* `sample_discrete(p, n, rng)` → array of `n` integer category indices for probabilities `p`, obtained
  by drawing `U = rng.random(n)` and returning the smallest `k` with `cumsum(p)[k] >= U`
  (`np.searchsorted` on the cumulative sums does exactly this).

Because the checks reproduce your draws with a fixed seed, use exactly one call `rng.random(n)` in each function.

In [ ]:
def sample_exponential(lam, n, rng):
    # YOUR CODE HERE  (u = rng.random(n); then transform)
    return None

def sample_discrete(p, n, rng):
    # YOUR CODE HERE  (cumulative sums + np.searchsorted)
    return None

**Check your answer:**

In [ ]:
try:
    xs = sample_exponential(2.0, 50_000, np.random.default_rng(1))
    m, v = float(np.mean(xs)), float(np.var(xs))
except Exception:
    xs, m, v = None, None, None
check('Exponential(2): sample mean within 2% of 1/2', m is not None and abs(m - 0.5) < 0.01)
check('Exponential(2): sample variance within 5% of 1/4', v is not None and abs(v - 0.25) < 0.0125)
check('Exponential(2): all samples positive', xs is not None and bool(np.all(np.asarray(xs) > 0)))
try:
    got = np.round(sample_exponential(2.0, 3, np.random.default_rng(7)), 4)
except Exception:
    got = None
check_hash('hidden: the first three Exponential(2) draws with seed 7', got, '8ec82f644f048596', decimals=4)
try:
    ks = np.asarray(sample_discrete([0.2, 0.5, 0.3], 50_000, np.random.default_rng(2)))
    freq = np.bincount(ks, minlength=3) / len(ks)
except Exception:
    ks, freq = None, None
check('discrete sampler: frequencies within 0.01 of p', freq is not None and bool(np.all(np.abs(freq - np.array([0.2, 0.5, 0.3])) < 0.01)))
check('discrete sampler returns integer indices in {0, 1, 2}', ks is not None and set(np.unique(ks).tolist()) <= {0, 1, 2})
try:
    got = np.asarray(sample_discrete([0.1, 0.2, 0.3, 0.4], 5, np.random.default_rng(7))).astype(int)
except Exception:
    got = None
check_hash('hidden: five categorical draws with seed 7', got, '0458632f0afa9a38')

## Exercise 2 — Rejection sampling for Beta(2, 5)  `★☆☆`

The Beta(2, 5) density is $p(x) = 30\,x(1-x)^4$ on $[0, 1]$; its maximum is at $x^* = 0.2$. Implement

* `beta25_pdf(x)` — the density (vectorised);
* `rejection_beta25(n_proposals, rng)` — rejection sampling with the uniform proposal $q = \text{Uniform}(0,1)$
  and the **tightest** valid envelope constant $M = p(0.2)$. Draw `x = rng.random(n_proposals)` first, then
  `u = rng.random(n_proposals)`, accept where `u <= p(x) / M`, and return the tuple
  `(accepted_samples, acceptance_rate)` with the rate as a float.

Only NumPy; do not use `scipy.stats.beta` to generate samples.

In [ ]:
def beta25_pdf(x):
    # YOUR CODE HERE
    return None

def rejection_beta25(n_proposals, rng):
    # YOUR CODE HERE  (x = rng.random(n); u = rng.random(n); accept where u <= p(x)/M)
    return None, None

**Check your answer:**

In [ ]:
try:
    got = float(beta25_pdf(0.2))
except Exception:
    got = None
check_close('density at the mode p(0.2) = 2.4576', got, 2.4576, tol=1e-9)
try:
    got = float(beta25_pdf(0.5))
except Exception:
    got = None
check_close('density at 0.5 agrees with scipy', got, float(stats.beta.pdf(0.5, 2, 5)), tol=1e-9)
try:
    samples, rate = rejection_beta25(200_000, np.random.default_rng(3))
    samples = np.asarray(samples, dtype=float); rate = float(rate)
except Exception:
    samples, rate = None, None
check('acceptance rate within 0.01 of 1/M = 0.4069', rate is not None and abs(rate - 1 / 2.4576) < 0.01)
check('number of accepted samples matches the rate', samples is not None and abs(len(samples) - rate * 200_000) < 1)
check('accepted samples have mean within 0.01 of 2/7', samples is not None and abs(samples.mean() - 2 / 7) < 0.01)
check('accepted samples have variance within 10% of 10/392', samples is not None and abs(samples.var() - 10 / 392) < 0.1 * 10 / 392)
check_hash('hidden: exact acceptance rate for seed 3', (round(rate, 4) if rate is not None else None), '5d4e93def6fb391c', decimals=4)

## Exercise 3 — Importance sampling for a Gaussian tail probability  `★★☆`

Estimate $\mu = P(X > a)$ for $X \sim \mathcal{N}(0, 1)$ with importance sampling from the shifted proposal
$q = \mathcal{N}(\text{shift}, 1)$. Implement

* `is_tail_estimate(a, n, shift, rng)` → the tuple `(estimate, fw)`, where the samples are
  `x = rng.normal(shift, 1.0, n)`, the weights are $w(x) = \phi(x)/\phi(x - \text{shift})$ (compute them from
  the formula $\exp\!\big(-x^2/2 + (x - \text{shift})^2/2\big)$ or with `scipy.stats.norm.pdf`), `fw` is the
  array of the $n$ weighted integrands $\mathbf{1}[x_i > a]\,w(x_i)$, and `estimate` is `fw.mean()` as a float;
* `effective_sample_size(w)` → $(\sum_i w_i)^2 / \sum_i w_i^2$ as a float.

Test case: `a = 4`, `n = 20_000`, `shift = 4`, `rng = np.random.default_rng(5)`; the truth is `stats.norm.sf(4)`.

In [ ]:
def is_tail_estimate(a, n, shift, rng):
    # YOUR CODE HERE  (x = rng.normal(shift, 1.0, n); weights phi(x)/phi(x - shift); fw = indicator * weights)
    return None, None

def effective_sample_size(w):
    # YOUR CODE HERE
    return None

**Check your answer:**

In [ ]:
truth = stats.norm.sf(4.0)
try:
    est, fw = is_tail_estimate(4.0, 20_000, 4.0, np.random.default_rng(5))
    est = float(est); fw = np.asarray(fw, dtype=float)
except Exception:
    est, fw = None, None
check('IS estimate of P(X > 4) within 5% of the truth', est is not None and abs(est - truth) / truth < 0.05)
check('fw has one entry per proposal and is zero where x <= 4 (about half of them)', fw is not None and fw.shape == (20_000,) and 0.4 < np.mean(fw == 0) < 0.6)
check_hash('hidden: the estimate for seed 5 (rounded to 7 decimals)', (round(est, 7) if est is not None else None), '19d2ac0cbe4f9d51', decimals=7)
try:
    got = effective_sample_size([0.1, 0.1, 0.1, 0.7])
except Exception:
    got = None
check_close('ESS of weights (0.1, 0.1, 0.1, 0.7) is 1/0.52', got, 1 / 0.52)
try:
    got = effective_sample_size(np.ones(50))
except Exception:
    got = None
check_close('ESS of 50 equal weights is 50', got, 50.0)
try:
    e = effective_sample_size(fw)
except Exception:
    e = None
check('ESS of the weighted integrands is between 3,000 and 4,500 (about 18% of n)', e is not None and 3_000 < e < 4_500)
try:
    e3 = effective_sample_size(is_tail_estimate(4.0, 20_000, 3.0, np.random.default_rng(5))[1])
    e5 = effective_sample_size(is_tail_estimate(4.0, 20_000, 5.0, np.random.default_rng(5))[1])
except Exception:
    e3, e5 = None, None
check('shifting the proposal to 3 or to 5 lowers the ESS compared with 4', e is not None and e3 is not None and e5 is not None and e3 < e and e5 < e)

## Exercise 4 — Stationary distributions and detailed balance  `★★☆`

Implement

* `stationary_distribution(P)` → the stationary distribution $\pi$ of the row-stochastic matrix `P`, solving
  $\pi = \pi P$ (hint: $\pi^\top$ is the eigenvector of $P^\top$ with eigenvalue 1; take the real part and
  normalise so the entries sum to 1), returned as a 1-D NumPy array;
* `satisfies_detailed_balance(P, pi, tol=1e-9)` → `True` if $\pi_i P_{ij} = \pi_j P_{ji}$ for all $i, j$
  (to tolerance `tol`), else `False`.

Test chains: the weather chain of the guide
`[[0.7, 0.2, 0.1], [0.3, 0.4, 0.3], [0.2, 0.3, 0.5]]`; the birth–death chain
`[[0.5, 0.5, 0], [0.25, 0.5, 0.25], [0, 0.5, 0.5]]`; and the cyclic chain
`[[0.1, 0.9, 0], [0, 0.1, 0.9], [0.9, 0, 0.1]]`.

In [ ]:
def stationary_distribution(P):
    # YOUR CODE HERE  (left eigenvector of P for eigenvalue 1, normalised to sum 1)
    return None

def satisfies_detailed_balance(P, pi, tol=1e-9):
    # YOUR CODE HERE
    return None

**Check your answer:**

In [ ]:
P_w = np.array([[0.7, 0.2, 0.1], [0.3, 0.4, 0.3], [0.2, 0.3, 0.5]])
P_q = np.array([[0.5, 0.5, 0.0], [0.25, 0.5, 0.25], [0.0, 0.5, 0.5]])
P_c = np.array([[0.1, 0.9, 0.0], [0.0, 0.1, 0.9], [0.9, 0.0, 0.1]])
try:
    pi_w = np.asarray(stationary_distribution(P_w), dtype=float).reshape(3)
except Exception:
    pi_w = None
check_close('weather chain: pi = (21, 13, 12)/46', pi_w, np.array([21, 13, 12]) / 46, tol=1e-6)
check_close('weather chain: pi P = pi', (pi_w @ P_w if pi_w is not None else None), pi_w, tol=1e-9)
try:
    pi_q = np.asarray(stationary_distribution(P_q), dtype=float).reshape(3)
except Exception:
    pi_q = None
check_hash('hidden: stationary distribution of the birth-death chain', (np.round(pi_q, 4) if pi_q is not None else None), 'f30b0b07dc722f68', decimals=4)
try:
    pi_c = np.asarray(stationary_distribution(P_c), dtype=float).reshape(3)
except Exception:
    pi_c = None
check_close('cyclic chain: uniform stationary distribution', pi_c, np.ones(3) / 3, tol=1e-6)
try:
    db = [bool(satisfies_detailed_balance(P, pi)) for P, pi in ((P_w, pi_w), (P_q, pi_q), (P_c, pi_c))]
except Exception:
    db = None
check('detailed balance: weather False, birth-death True, cyclic False', db == [False, True, False])
try:
    got = satisfies_detailed_balance(np.eye(2), np.array([0.3, 0.7]))
except Exception:
    got = None
check('the identity chain satisfies detailed balance for any pi', got is True)

## Exercise 5 — Random-walk Metropolis–Hastings for a Gamma target  `★★☆`

The target is Gamma(shape 3, rate 1), known only up to a constant: $\log\tilde p(x) = 2\ln x - x$ for $x > 0$
and $-\infty$ otherwise. Implement

* `log_gamma3(x)` — the unnormalised log density for a scalar `x` (return `-np.inf` for `x <= 0`);
* `acceptance_probability(log_target, x, prop)` → $\min\{1, \tilde p(\text{prop})/\tilde p(x)\}$ for a symmetric
  proposal, computed from the log target;
* `random_walk_mh(log_target, x0, step, n, rng)` → `(chain, acceptance_rate)`: `n` iterations starting from
  `x0`; at each iteration draw `prop = x + step * rng.standard_normal()`, then `u = rng.random()`, accept
  if `np.log(u) < log_target(prop) - log_target(x)`, and record the current state (whether or not the move was
  accepted) in a NumPy array of length `n`. Exactly these two `rng` calls per iteration, in this order.

Test: `random_walk_mh(log_gamma3, 3.0, 2.5, 30_000, np.random.default_rng(11))`, discarding the first 2,000
samples as burn-in. Gamma(3, 1) has mean 3 and variance 3.

In [ ]:
def log_gamma3(x):
    # YOUR CODE HERE
    return None

def acceptance_probability(log_target, x, prop):
    # YOUR CODE HERE
    return None

def random_walk_mh(log_target, x0, step, n, rng):
    # YOUR CODE HERE  (per iteration: prop = x + step*rng.standard_normal(); u = rng.random())
    return None, None

**Check your answer:**

In [ ]:
try:
    got = [float(log_gamma3(1.0)), float(log_gamma3(2.0)), float(log_gamma3(-1.0))]
    ok_log = abs(got[0] + 1) < 1e-12 and abs(got[1] - (2 * np.log(2) - 2)) < 1e-12 and got[2] == -np.inf
except Exception:
    ok_log = False
check('log target: log p(1) = -1, log p(2) = 2 ln 2 - 2, log p(-1) = -inf', ok_log)
try:
    a1 = acceptance_probability(log_gamma3, 1.0, 2.0)
    a2 = acceptance_probability(log_gamma3, 2.0, 3.0)
    a3 = acceptance_probability(log_gamma3, 2.0, 6.0)
except Exception:
    a1 = a2 = a3 = None
check_close('uphill move 1 -> 2 (towards the mode at 2) is accepted with probability 1', a1, 1.0)
check_hash('hidden: acceptance probability of the downhill move 2 -> 3', (round(a2, 6) if a2 is not None else None), 'd8a4eda59292fe75', decimals=6)
check_hash('hidden: acceptance probability of the downhill move 2 -> 6', (round(a3, 6) if a3 is not None else None), '1cd877705cdc9115', decimals=6)
try:
    chain, rate = random_walk_mh(log_gamma3, 3.0, 2.5, 30_000, np.random.default_rng(11))
    chain = np.asarray(chain, dtype=float); rate = float(rate); post = chain[2000:]
except Exception:
    chain, rate, post = None, None, None
check('chain has length 30,000 and stays positive', chain is not None and chain.shape == (30_000,) and bool(np.all(chain > 0)))
check('acceptance rate between 0.45 and 0.65 for step 2.5', rate is not None and 0.45 < rate < 0.65)
check('posterior mean within 0.1 of 3', post is not None and abs(post.mean() - 3) < 0.1)
check('posterior variance within 0.3 of 3', post is not None and abs(post.var() - 3) < 0.3)
check_hash('hidden: exact acceptance rate for seed 11, step 2.5', (round(rate, 4) if rate is not None else None), 'b66fa6065864db1b', decimals=4)
try:
    rate_small = random_walk_mh(log_gamma3, 3.0, 0.3, 30_000, np.random.default_rng(11))[1]
    rate_large = random_walk_mh(log_gamma3, 3.0, 12.0, 30_000, np.random.default_rng(11))[1]
except Exception:
    rate_small = rate_large = None
check('tiny step accepts > 85%, huge step accepts < 25%', rate_small is not None and rate_large is not None and rate_small > 0.85 and rate_large < 0.25)

## Exercise 6 — Dimension kills rejection sampling; the direction of the KL matters  `★★☆`

1. A colleague proposes to sample a 200-dimensional posterior by rejection sampling with a Gaussian proposal
   whose covariance is "about right, within 10% in each direction". Estimate the acceptance rate and explain
   why no amount of tuning will save the plan.
2. Variational inference minimises $\text{KL}(q \,\|\, p)$ (reverse KL). Suppose the true posterior $p$ is a
   50/50 mixture of two well-separated Gaussians and $q$ is a single Gaussian. Describe the $q$ that minimises
   the reverse KL and the $q$ that would minimise the forward KL $\text{KL}(p \,\|\, q)$, and explain why VI
   uses the reverse direction anyway.

<details><summary><b>Show answer</b></summary>

1. For target $\mathcal{N}(0, I_d)$ and proposal $\mathcal{N}(0, \sigma^2 I_d)$ with $\sigma = 1.1$ (the proposal
   must be *wider* than the target or no finite envelope exists), the tightest constant is
   $M = \sup_x p(x)/q(x) = \sigma^d$, so the acceptance rate is $\sigma^{-d} = 1.1^{-200} \approx 5 \times 10^{-9}$
   — one accepted sample per 200 million proposals. The failure is structural: a per-coordinate mismatch of
   $\sigma$ becomes $\sigma^d$ in the density ratio, and even $\sigma = 1.01$ gives $1.01^{-200} \approx 0.14$ only
   because the proposal is then essentially the target, which we cannot construct without already knowing the
   posterior. Any error in the proposal is amplified exponentially in $d$, which is why high-dimensional
   sampling uses Markov chains (local moves, no global envelope) or variational methods.
2. The reverse KL $\text{KL}(q \,\|\, p) = \mathbb{E}_q[\log q - \log p]$ is huge wherever $q$ puts mass but $p$ is
   tiny, so the optimal single Gaussian sits on *one* of the two modes, matching its width (**mode-seeking**,
   variance underestimated: it ignores half the posterior). The forward KL $\text{KL}(p \,\|\, q) = \mathbb{E}_p[\log p - \log q]$
   penalises $q$ wherever $p$ has mass and $q$ does not, so its minimiser is a broad Gaussian covering both modes
   with the correct overall mean and inflated variance (**mass-covering**). VI uses the reverse direction because
   its expectation is under $q$, which we can sample and differentiate through, and because it leads to the
   ELBO, which needs only the unnormalised $p(x, z)$; the forward KL requires expectations under the unknown
   posterior — the very thing we are trying to compute.

</details>

## Exercise 7 — The VAE's closed-form KL, verified by Monte Carlo  `★★☆`

For a diagonal Gaussian encoder output $q = \mathcal{N}(\mu, \operatorname{diag}\sigma^2)$ and prior $p = \mathcal{N}(0, I)$,
implement

* `kl_gaussian_to_standard(mu, log_sigma)` → $\text{KL}(q \,\|\, p) = \sum_j \frac12\big(\mu_j^2 + \sigma_j^2 - 1 - \log\sigma_j^2\big)$
  as a float, where `log_sigma` holds $\log\sigma_j$ (the quantity a VAE encoder outputs);
* `kl_monte_carlo(mu, log_sigma, n, rng)` → the Monte Carlo estimate $\frac1n\sum_i[\log q(z_i) - \log p(z_i)]$
  from `n` reparameterised samples `z = mu + sigma * rng.standard_normal((n, k))`, using `scipy.stats.norm.logpdf`
  for the two log densities (sum over the `k` coordinates), as a float.

Test: $\mu = (0.5, -1.0)$, $\sigma = (0.8, 0.5)$ — the worked example of the guide, where the KL is $0.9863$.

In [ ]:
def kl_gaussian_to_standard(mu, log_sigma):
    # YOUR CODE HERE
    return None

def kl_monte_carlo(mu, log_sigma, n, rng):
    # YOUR CODE HERE  (z = mu + sigma * rng.standard_normal((n, k)); average of log q(z) - log p(z))
    return None

**Check your answer:**

In [ ]:
try:
    got = kl_gaussian_to_standard([0.5, -1.0], np.log([0.8, 0.5]))
except Exception:
    got = None
check_close('KL for mu = (0.5, -1), sigma = (0.8, 0.5) is 0.98629', got, 0.986290731874155, tol=1e-6)
try:
    got = kl_gaussian_to_standard([0.0, 0.0], [0.0, 0.0])
except Exception:
    got = None
check_close('KL is zero when q equals the prior', got, 0.0, tol=1e-12)
try:
    got = kl_gaussian_to_standard([1.0, 2.0, -0.5], [0.0, -0.5, 0.3])
except Exception:
    got = None
check_hash('hidden: KL for a 3-D case', (round(got, 5) if got is not None else None), '8f615740987b0b49', decimals=5)
try:
    got = kl_gaussian_to_standard([0.0], [np.log(0.05)])
except Exception:
    got = None
check('a near-deterministic code (sigma = 0.05) costs more than 2 nats', got is not None and got > 2.0)
try:
    mc = kl_monte_carlo([0.5, -1.0], np.log([0.8, 0.5]), 200_000, np.random.default_rng(9))
except Exception:
    mc = None
check('Monte Carlo KL within 0.01 of the closed form', mc is not None and abs(mc - 0.986290731874155) < 0.01)
check_hash('hidden: the Monte Carlo estimate for seed 9 (3 decimals)', (round(mc, 3) if mc is not None else None), 'a8615c3feb3f99e3', decimals=3)

## Exercise 8 — The diffusion forward process in closed form  `★★☆`

With $\alpha_t = 1 - \beta_t$ and $\bar\alpha_t = \prod_{s \le t}\alpha_s$, the forward marginal is
$q(x_t \mid x_0) = \mathcal{N}(\sqrt{\bar\alpha_t}\,x_0,\ 1 - \bar\alpha_t)$ (one dimension). Implement, with `t`
**1-based** (so `t = 1` uses $\beta_1$ only):

* `alpha_bar(betas)` → the array $(\bar\alpha_1, \dots, \bar\alpha_T)$;
* `forward_marginal(x0, t, betas)` → the tuple `(mean, variance)` of $q(x_t \mid x_0)$ as floats;
* `noisy_sample(x0, t, betas, eps)` → $\sqrt{\bar\alpha_t}x_0 + \sqrt{1 - \bar\alpha_t}\,\varepsilon$ for a given noise value `eps`;
* `implied_noise(x0, t, betas, x_t)` → the $\varepsilon$ that produced `x_t` from `x0`, i.e. $(x_t - \sqrt{\bar\alpha_t}x_0)/\sqrt{1 - \bar\alpha_t}$ — the regression target of the diffusion loss.

Tests: the guide's schedule $\beta = (0.1, 0.2, 0.3)$ with $x_0 = 2$, and the DDPM schedule
`np.linspace(1e-4, 0.02, 1000)`.

In [ ]:
def alpha_bar(betas):
    # YOUR CODE HERE
    return None

def forward_marginal(x0, t, betas):
    # YOUR CODE HERE  (t is 1-based)
    return None, None

def noisy_sample(x0, t, betas, eps):
    # YOUR CODE HERE
    return None

def implied_noise(x0, t, betas, x_t):
    # YOUR CODE HERE
    return None

**Check your answer:**

In [ ]:
b3 = [0.1, 0.2, 0.3]
try:
    got = np.asarray(alpha_bar(b3), dtype=float)
except Exception:
    got = None
check_close('alpha_bar for beta = (0.1, 0.2, 0.3) is (0.9, 0.72, 0.504)', got, [0.9, 0.72, 0.504], tol=1e-12)
try:
    m, v = forward_marginal(2.0, 3, b3)
except Exception:
    m, v = None, None
check_close('marginal mean at t = 3 from x0 = 2 is 2 sqrt(0.504)', m, 2 * np.sqrt(0.504), tol=1e-9)
check_close('marginal variance at t = 3 is 0.496', v, 0.496, tol=1e-9)
try:
    got = noisy_sample(2.0, 3, b3, -0.5)
except Exception:
    got = None
check_close('noisy sample with eps = -0.5', got, 2 * np.sqrt(0.504) - 0.5 * np.sqrt(0.496), tol=1e-9)
try:
    got = implied_noise(2.0, 3, b3, 1.0)
except Exception:
    got = None
check_hash('hidden: implied noise for x_3 = 1.0 (the guide example)', (round(got, 4) if got is not None else None), 'b3bc4b24fef0f7ea', decimals=4)
try:
    got = implied_noise(2.0, 3, b3, noisy_sample(2.0, 3, b3, 0.7))
except Exception:
    got = None
check_close('implied_noise inverts noisy_sample', got, 0.7, tol=1e-9)
lin = np.linspace(1e-4, 0.02, 1000)
try:
    ab = np.asarray(alpha_bar(lin), dtype=float)
    abT = float(ab[-1]); snr500 = float(ab[499] / (1 - ab[499]))
except Exception:
    ab, abT, snr500 = None, None, None
check('DDPM linear schedule: alpha_bar_T below 1e-4 (pure noise at the end)', abT is not None and abT < 1e-4)
check_hash('hidden: signal-to-noise ratio at t = 500 (3 decimals)', (round(snr500, 3) if snr500 is not None else None), '70a47fadae9fed82', decimals=3)

## Exercise 9 — The score of a Gaussian mixture and Langevin sampling  `★★★`

For the mixture $p(x) = \sum_k w_k\,\mathcal{N}(x; m_k, s_k^2)$ implement

* `mixture_logpdf(x, w, m, s)` → $\log p(x)$, vectorised over an array `x` (use the log-sum-exp trick);
* `mixture_score(x, w, m, s)` → $\nabla_x\log p(x) = \sum_k r_k(x)\,\frac{-(x - m_k)}{s_k^2}$ with responsibilities
  $r_k(x) = w_k\mathcal{N}(x; m_k, s_k^2)/p(x)$, vectorised over `x`;
* `langevin(score_fn, x0, eps, n_steps, rng)` → the particles after `n_steps` updates
  $x \leftarrow x + \frac{\epsilon}{2}\,\text{score}(x) + \sqrt{\epsilon}\,\xi$ with `xi = rng.standard_normal(x.shape)`
  (one such call per step), starting from the array `x0`.

Test mixture: `w = [0.3, 0.7]`, `m = [-1.0, 2.0]`, `s = [0.5, 1.0]`. The score is checked against central
finite differences of your `mixture_logpdf` **and** against a hidden reference.

In [ ]:
def mixture_logpdf(x, w, m, s):
    # YOUR CODE HERE
    return None

def mixture_score(x, w, m, s):
    # YOUR CODE HERE  (responsibilities times the per-component scores)
    return None

def langevin(score_fn, x0, eps, n_steps, rng):
    # YOUR CODE HERE  (x <- x + eps/2 * score(x) + sqrt(eps) * rng.standard_normal(x.shape))
    return None

**Check your answer:**

In [ ]:
w, m, s = [0.3, 0.7], [-1.0, 2.0], [0.5, 1.0]
pts = np.array([-1.5, 0.0, 0.8, 3.0])
try:
    lp = np.asarray(mixture_logpdf(pts, w, m, s), dtype=float)
    ref = np.log(0.3 * stats.norm.pdf(pts, -1, 0.5) + 0.7 * stats.norm.pdf(pts, 2, 1))
except Exception:
    lp, ref = None, None
check_close('log density matches scipy at four points', lp, ref, tol=1e-9)
try:
    got = np.asarray(mixture_score(np.array([0.5, 2.0]), [1.0], [0.0], [1.0]), dtype=float)
except Exception:
    got = None
check_close('single standard normal: score is -x', got, [-0.5, -2.0], tol=1e-9)
try:
    sc = np.asarray(mixture_score(pts, w, m, s), dtype=float)
    h = 1e-5
    fd = (np.asarray(mixture_logpdf(pts + h, w, m, s)) - np.asarray(mixture_logpdf(pts - h, w, m, s))) / (2 * h)
except Exception:
    sc, fd = None, None
check_close('score matches finite differences of the log density', sc, fd, tol=1e-5)
check_hash('hidden: score at the four test points (4 decimals)', (np.round(sc, 4) if sc is not None else None), '16be280a0c52cd85', decimals=4)
try:
    x_final = np.asarray(langevin(lambda x: mixture_score(x, w, m, s),
                                  4 * np.random.default_rng(1).standard_normal(5000), 0.05, 600, np.random.default_rng(1)), dtype=float)
    if x_final.shape != (5000,):
        raise ValueError("wrong shape")
    frac = float(np.mean(x_final > 0.5)); mean_right = float(x_final[x_final > 0.5].mean())
except Exception:
    x_final, frac, mean_right = None, None, None
check('Langevin: about 70% of particles end in the right-hand mode', frac is not None and 0.64 < frac < 0.76)
check('Langevin: right-hand particles average near 2', mean_right is not None and abs(mean_right - 2.0) < 0.15)
check_hash('hidden: exact fraction in the right mode for seed 1 (3 decimals)', (round(frac, 3) if frac is not None else None), 'a4b2c3a0b8e876d5', decimals=3)

## Exercise 10 — Mini-project: Bayesian inference for a Poisson rate with Metropolis–Hastings  `★★★`

A hospital ward records $y = (3, 5, 2, 4, 6, 3, 4, 5, 2, 3)$ emergency admissions on ten consecutive nights,
modelled as $y_i \sim \text{Poisson}(\lambda)$ with prior $\lambda \sim \text{Gamma}(\text{shape } 2, \text{rate } 1)$.
Because the model is conjugate (Module 11), the exact posterior is $\text{Gamma}(2 + \sum y_i,\ 1 + n) = \text{Gamma}(39, 11)$
— which lets you *grade your own sampler*.

**Tasks** (use `rng = np.random.default_rng(2024)`):

1. Write the unnormalised log posterior $\log\tilde p(\lambda) = (a_0 - 1)\ln\lambda - b_0\lambda + \sum_i y_i\ln\lambda - n\lambda$
   for $\lambda > 0$ (and $-\infty$ otherwise) and a random-walk MH sampler.
2. Run 20,000 iterations from the deliberately poor start $\lambda_0 = 1$ for step sizes
   $s \in \{0.05, 0.3, 1, 3, 10\}$. For each: acceptance rate, trace plot, autocorrelation function, integrated
   autocorrelation time $\tau$ and $\text{ESS} = n/\tau$ after discarding 2,000 burn-in iterations.
3. Pick the step with the largest ESS and compare its samples with the exact Gamma(39, 11) posterior:
   histogram against the density, posterior mean and sd, the 2.5%/50%/97.5% quantiles, and
   $P(\lambda > 4 \mid y)$ with a Monte Carlo standard error computed from the ESS (not from $n$).
4. Show why burn-in matters: compare the mean of the first 200 iterations of the smallest-step chain with the
   exact posterior mean.
5. Run four shorter chains (5,000 iterations) from the starts $\{0.5, 2, 5, 9\}$ with your best step and compute
   $\hat R = \sqrt{\hat V / W}$, where $W$ is the mean within-chain variance, $B = n\cdot\operatorname{Var}(\text{chain means})$
   and $\hat V = \frac{n-1}{n}W + \frac{B}{n}$.

**Rubric — you have mastered this when you can:**

* The log posterior is correct (prior **and** likelihood, $-\infty$ for $\lambda \le 0$) and the sampler accepts with
  probability $\min\{1, \tilde p(\lambda')/\tilde p(\lambda)\}$ computed in log space, recording the current state after
  a rejection.
* Acceptance rate, $\tau$ and ESS are reported for every step size, and the best step (largest ESS, acceptance
  roughly 0.3–0.6, close to $2.4\times$ the posterior sd) is chosen by ESS rather than by acceptance rate.
* The MH posterior mean and sd agree with the exact $39/11 = 3.545$ and $\sqrt{39}/11 = 0.568$ to within Monte
  Carlo error, and the histogram lies on the Gamma(39, 11) density.
* The standard error of $P(\lambda > 4 \mid y)$ uses the ESS, and the exact value from `scipy.stats.gamma.sf` is
  within about two standard errors.
* The burn-in comparison shows the early iterations of the slow chain are biased toward the start, and
  $\hat R$ from four dispersed chains is close to 1.
* All figures have titles and axis labels.

In [ ]:
# Your mini-project workspace

## Score

In [ ]:
summary()